# A Noise-Aware Quantum Algorithm for Credit Valuation Adjustments on Real Quantum Hardware
**ArXivist-generated reproduction notebook**

Paper: [arXiv:2607.12990](https://arxiv.org/abs/2607.12990)
Generated: 2026-07-16

This notebook walks through the key components of the implementation, runs a
small-scale (few-qubit) end-to-end pipeline, and verifies that the setup
matches the paper's reported behavior on a toy instance. No GPU or internet
access is required -- everything here runs on CPU via Qiskit's statevector
simulator.

In [ ]:
# Check Python version and key dependencies (no GPU needed -- all circuits are <=9 qubits,
# so CPU statevector simulation is fast and is what the paper itself uses for training).
import sys
print(f"Python: {sys.version}")

try:
    import qiskit
    print(f"Qiskit: {qiskit.__version__}")
except ImportError:
    print("Qiskit not found -- run the Installation cell below first.")

try:
    import numpy as np
    print(f"NumPy: {np.__version__}")
except ImportError:
    print("NumPy not found -- run the Installation cell below first.")

try:
    import torch
    print(f"PyTorch: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()} (not required for this paper)")
except ImportError:
    print("PyTorch not found (only needed if you extend training/trainer.py with torch.optim).")


In [ ]:
# Install the project in editable mode (run once)
import subprocess
result = subprocess.run(["pip", "install", "-e", ".."], capture_output=True, text=True)
print(result.stdout[-2000:] if result.returncode == 0 else result.stderr[-2000:])


## Paper Overview

**Problem**: Credit Valuation Adjustment (CVA) requires repeatedly estimating
risk-neutral expectations via Monte Carlo, whose error shrinks only as
$O(N^{-1/2})$. Quantum Amplitude Estimation (QAE) can in principle achieve
$O(N^{-1})$ -- a quadratic speedup -- but only if (a) the CVA functional can be
encoded faithfully into a quantum circuit's success amplitude, and (b) the
amplitude-estimation algorithm survives real hardware noise.

**Core innovation -- CABIQAE**: the paper's central contribution is
*Contrast-Aware Bayesian Iterative Quantum Amplitude Estimation*. Instead of
treating Grover-amplified circuits as noiselessly following the ideal
oscillation $\sin^2((2k{+}1)\theta)$, CABIQAE folds an experimentally
calibrated contrast-decay model directly into the Bayesian likelihood:

$$p_{obs}(\vartheta,k) = b + c(k)\left(\sin^2(K\vartheta)-b\right), \qquad c(k)=c_0 e^{-K/\tau_c}, \quad K=2k+1$$

This lets the Grover-depth scheduler avoid wasting queries on deep, low-contrast
circuits that noiseless BIQAE would otherwise (over-)trust.

**Pipeline** (see `README.md` for the full section-to-module map):
1. Classical market calibration & finite-grid CVA benchmark (`finance/`)
2. Quantum encoding: QCBM state prep + 3 controlled-rotation blocks (`circuits/`)
3. CABIQAE / BIQAE / BAE / DCS amplitude estimation (`estimation/`)
4. Formal error-budget decomposition (`evaluation/error_decomposition.py`)


## Component 1: QCBM State Preparation ($G_\theta$)

Encodes the joint time-market probability distribution into a quantum state's
amplitudes (Eq. 33, Section 3.2.3, Figure 3):

$$G_P|0\rangle^{\otimes(m+n)} = \sum_{i,j} \sqrt{P_{i,j}}\,|i\rangle_T|j\rangle_S$$

trained by minimising the cross-entropy loss $\mathcal{L}_{QCBM}(\theta) = -\sum_x P_{target}(x)\log(P_\theta(x)+\epsilon_{num})$.

In [ ]:
import numpy as np
from quantum_cva.circuits import QCBMStatePreparation

# Toy instance: m=1 time qubit, n=1 asset qubit (paper's real instance uses m=2, n=4)
qcbm = QCBMStatePreparation(num_time_qubits=1, num_asset_qubits=1)
circuit = qcbm.build_circuit(num_layers=1)

rng = np.random.default_rng(0)
params = rng.uniform(-np.pi, np.pi, circuit.num_parameters)
dist = qcbm.born_distribution(circuit, params)

print(f"Circuit qubits: {circuit.num_qubits}")
print(f"Circuit parameters: {circuit.num_parameters}")
print(f"Born distribution shape: {dist.shape}  (expected: (4,) = 2**(1+1))")
print(f"Sums to 1: {np.isclose(dist.sum(), 1.0)}")


## Component 2: Controlled-Rotation Circuit Ansatz (CRCA) blocks

Three CRCA blocks encode deterministic scalar functions into ancilla-qubit
amplitudes (Section 3.2.3, Figures 4-5): $R_p$ (discount factor), $R_q$
(default probability) use a compact native-tree topology conditioned only on
the time register; $R_v$ (positive exposure) uses a "snake" topology since it
must condition on the *full* time-market register (exposure is non-linear and
non-separable across time and asset coordinates):

$$|0\rangle \longmapsto \sqrt{1-F_\phi(x)}\,|0\rangle + \sqrt{F_\phi(x)}\,|1\rangle$$

In [ ]:
from quantum_cva.circuits import NativeTreeCRCA, SnakeCRCA

r_p = NativeTreeCRCA(num_time_qubits=1)
rp_circuit = r_p.build_circuit(num_layers=1)
rp_params = rng.uniform(-np.pi, np.pi, rp_circuit.num_parameters)
f_p_0 = r_p.conditional_probability(rp_circuit, rp_params, time_index=0)
print(f"R_p circuit qubits: {rp_circuit.num_qubits} (2 time-conditioning + 1 ancilla for m=1)")
print(f"F_phi(i=0) = {f_p_0:.4f}  (should be in [0, 1])")

r_v = SnakeCRCA(num_time_qubits=1, num_asset_qubits=1)
rv_circuit = r_v.build_circuit(num_layers=1)
rv_params = rng.uniform(-np.pi, np.pi, rv_circuit.num_parameters)
f_v = r_v.conditional_probability(rv_circuit, rv_params, time_index=0, asset_index=(0,), asset_qubits_per_underlying=1)
print(f"R_v circuit qubits: {rv_circuit.num_qubits}")
print(f"F_phi(i=0, j=(0,)) = {f_v:.4f}  (should be in [0, 1])")


## Component 3: Full CVA Oracle Assembly

Composes $G_\theta, R_v, R_p, R_q$ into the full encoding unitary
$A_\Theta = R_q R_p R_v G_\theta$ (Section 2.2.1, Eq. 33-37, Figure 1), and
projects onto the marked ancilla subspace $\Pi_{111}$ to read out the encoded
CVA amplitude $a_{CVA} = \langle\xi|\Pi_{111}|\xi\rangle$.

In [ ]:
from quantum_cva.circuits import CVAOracle

g_circuit = qcbm.build_circuit(1).assign_parameters(params)
rv_bound = rv_circuit.assign_parameters(rv_params)
rp_bound = rp_circuit.assign_parameters(rp_params)

r_q = NativeTreeCRCA(num_time_qubits=1)
rq_circuit = r_q.build_circuit(num_layers=1)
rq_params = rng.uniform(-np.pi, np.pi, rq_circuit.num_parameters)
rq_bound = rq_circuit.assign_parameters(rq_params)

oracle = CVAOracle(num_register_qubits=2, num_ancillas=3)
a_theta = oracle.assemble(g_circuit, rv_bound, rp_bound, rq_bound)

a_cva = oracle.marked_amplitude_statevector(a_theta)
print(f"Full oracle A_Theta qubits: {a_theta.num_qubits}  (2 register + 3 ancillas)")
print(f"Encoded marked amplitude a_CVA = {a_cva:.6f}  (in [0, 1] as expected)")


## Component 4: CABIQAE -- the paper's core contribution

Runs the adaptive Bayesian amplitude-estimation loop (Appendix A.3, Algorithm
1) against the oracle we just built, comparing the noiseless ideal regime to
a synthetic "hardware-replay" regime with contrast decay ($\tau_c$ small ->
signal saturates quickly, exactly the paper's reported behavior for the full
CVA circuit, Table 18: $\tau_c \approx 2.2$ vs. $\tau_c \approx 34$ for the
much shallower validation circuit).

In [ ]:
from quantum_cva.estimation import CABIQAE, BIQAE

theta_true = np.arcsin(np.sqrt(a_cva))
est_rng = np.random.default_rng(1)

def ideal_executor(k, n_shots):
    K = 2 * k + 1
    p = np.sin(K * theta_true) ** 2
    return int(est_rng.binomial(n_shots, p)), n_shots

cabiqae = CABIQAE(c0=1.0, tau_c=1e12, b=0.5, rho_min=2.0)  # ideal (no contrast decay)
result_ideal = cabiqae.estimate(ideal_executor, epsilon=0.02, alpha=0.10, n_batch=128, max_stages=40)
print("=== Ideal (noiseless) regime ===")
print(result_ideal)
print(f"True amplitude: {a_cva:.6f}")
print(f"Relative error: {abs(result_ideal.a_hat - a_cva) / a_cva:.4%}")


In [ ]:
# Hardware-replay regime: short contrast length tau_c=2.2 (matching the paper's
# full CVA oracle calibration, Table 18) -- CABIQAE adapts by avoiding deep,
# uninformative Grover circuits; a noise-naive BIQAE does not.
est_rng2 = np.random.default_rng(2)

def hw_replay_executor(k, n_shots, _c0=0.67, _tau_c=2.2, _b=0.15):
    K = 2 * k + 1
    q_k = np.sin(K * theta_true) ** 2
    c_k = _c0 * np.exp(-K / _tau_c)
    p_hw = np.clip(_b + c_k * (q_k - _b), 0, 1)
    return int(est_rng2.binomial(n_shots, p_hw)), n_shots

cabiqae_hw = CABIQAE(c0=0.67, tau_c=2.2, b=0.15, rho_min=2.0)
biqae_hw = BIQAE(rho_min=2.0)  # noise-naive baseline

result_cabiqae = cabiqae_hw.estimate(hw_replay_executor, epsilon=0.05, alpha=0.10, n_batch=128, max_stages=40)
result_biqae = biqae_hw.estimate(hw_replay_executor, epsilon=0.05, alpha=0.10, n_batch=128, max_stages=40)

print("=== Hardware-replay regime (short contrast length tau_c=2.2) ===")
print(f"CABIQAE: {result_cabiqae}  (K_max={2*result_cabiqae.max_k+1})")
print(f"BIQAE:   {result_biqae}  (K_max={2*result_biqae.max_k+1})")
print()
print("Note: CABIQAE's contrast-aware scheduler keeps K_max small and stays")
print("informative; noise-naive BIQAE tends to push to much larger K and can")
print("degrade -- qualitatively matching the paper's Table 6 finding.")


## Mini end-to-end training demonstration (synthetic data, ~10 iterations)

Trains a QCBM against a random target distribution using exact parameter-shift
gradients (all gates here are Pauli rotations, so parameter-shift gives exact
analytic gradients -- see `training/trainer.py`).

In [ ]:
import functools
from quantum_cva.training import VariationalTrainer

# Tiny synthetic target distribution (no downloads needed)
target_rng = np.random.default_rng(3)
target_dist = target_rng.dirichlet(np.ones(4))  # random distribution over 2**(1+1)=4 states
print(f"Synthetic target distribution: {target_dist.round(4)}")

qcbm_demo = QCBMStatePreparation(num_time_qubits=1, num_asset_qubits=1)
demo_circuit = qcbm_demo.build_circuit(num_layers=2)  # 2 layers for enough expressivity in this toy demo

loss_fn = functools.partial(
    qcbm_demo.cross_entropy_loss, demo_circuit, target_dist=target_dist, eps_num=1e-8
)

trainer = VariationalTrainer(optimizer="adam", learning_rate=0.01, seed=0)
trained_params, history = trainer.train_qcbm(
    loss_fn, demo_circuit.num_parameters, num_iterations=30, log_every_n_steps=5
)

print(f"\nLoss decreased: {history[0]:.4f} -> {history[-1]:.4f}")
assert history[-1] < history[0], "Training should decrease the loss"
final_dist = qcbm_demo.born_distribution(demo_circuit, trained_params)
print(f"Final Born distribution: {final_dist.round(4)}")
print(f"Target distribution:     {target_dist.round(4)}")


## Paper's Reported Results (from the SIR)

The numbers below are directly transcribed from the paper's Tables 1 and 6.
Reproducing them exactly requires the *full* six-qubit instance (`configs/config.yaml`
defaults), R=300 trajectories, and (for the hardware-replay numbers) either
real IBM Quantum access or the paper's own published calibration parameters
(already wired into `configs/config.yaml`'s `evaluation.contrast_*` fields).

In [ ]:
paper_results = {
    "CVA_cont_MC (continuous Monte Carlo benchmark)": 1.091,
    "CVA_tab_Delta (finite-grid benchmark, n=4)": 0.522,
    "CVA_SV(Theta) (trained circuit, statevector)": 0.670,
    "CABIQAE median relative CVA error (noiseless)": 0.000227,
    "CABIQAE median relative CVA error (hardware-replay)": 0.0515,
    "BIQAE median relative CVA error (noiseless)": 0.000242,
    "BIQAE median relative CVA error (hardware-replay)": 0.892,
    "CABIQAE classical runtime, validation circuit (s)": 1.25,
    "BAE classical runtime, validation circuit (s)": 45.72,
}
print("Paper's claimed results (Tables 1, 6, 7):")
for k, v in paper_results.items():
    print(f"  {k}: {v}")

print()
print("To reproduce these at full scale, run:")
print("  python train.py --config configs/config.yaml")
print("  python evaluate.py --config configs/config.yaml --regime hardware-replay --num-trajectories 300")
print("Then compare your results/comparison_*.json against the values above")
print("(Stage 6 of the ArXivist pipeline: Results Comparator).")


## What to do next

1. **Full training** (uses the real six-qubit paper instance):
   `python train.py --config configs/config.yaml --output-dir checkpoints/`
2. **Full evaluation**:
   `python evaluate.py --config configs/config.yaml --regime hardware-replay --num-trajectories 300`
3. **Compare your results** against the paper's published numbers using
   ArXivist's Results Comparator (Stage 6) -- feed it your `results/comparison_*.json`.

**Top 3 implementation assumptions to be aware of** (see `sir.json` for the full list):

1. *Optimizer/learning-rate/iteration counts* for QCBM/CRCA training are not named in
   the paper -- ASSUMED Adam + parameter-shift gradients (confidence 0.4).
2. *Q-CTRL's Performance Management* internals are a documented black box --
   this repo uses a labeled fallback that replays the paper's own published
   calibration statistics instead of fabricating hardware behavior (confidence 0.3).
3. *Market data* (LSEG Workspace) is proprietary -- a synthetic fallback is used
   by default; see `data/README_data.md` (evaluation_protocol confidence 0.88
   overall, but this specific data-access caveat applies).
